# TerraCast · Ensemble Yield Prediction + MLflow 3

Corn / Soybeans split model with:
- State / County fixed effects
- Ensemble: LightGBM + Random Forest (simple average)
- MLflow 3 experiment tracking (LoggedModel, Dataset linking)
- 5-Fold CV anomaly detection

**Run in Databricks** (Spark available)

In [ ]:
%pip install -U 'mlflow>=3.1' lightgbm geopandas scikit-learn shap pygris

In [ ]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold, train_test_split

import mlflow
import mlflow.lightgbm
import mlflow.sklearn
from mlflow.entities import Dataset

# Spark session in Databricks; None locally
try:
    spark = globals().get("spark", None)
except NameError:
    spark = None

print("MLflow version:", mlflow.__version__)

## Config

In [ ]:
# --- Edit these ---
MLFLOW_EXPERIMENT_ID = "2445807330421530"   # Databricks MLflow experiment ID
MERGED_TABLE         = "workspace.default.merged"  # Spark table (Databricks)
CSV_PATH             = None                 # local CSV fallback, e.g. Path("/path/to/merged.csv")

# Year-based train/test split (prevents time leakage)
TEST_START_YEAR = 2021  # years >= 2021 are test, < 2021 are train

# Region map (state → region) — based on USDA regional classification
# Northeast merged into adjacent regions to avoid data sparsity:
#   NY, PA, VT, CT, MA, ME, NH, RI, NJ → Midwest (northern climate)
#   MD, DE → South (mid-Atlantic climate)
REGION_MAP = {
    "Midwest": {
        "IA", "IL", "IN", "MI", "MN", "MO", "OH", "WI",   # Corn Belt core
        "NY", "PA", "VT", "CT", "MA", "ME", "NH", "RI", "NJ",  # Northeast (northern)
    },
    "Plains": {
        "CO", "KS", "MT", "ND", "NE", "OK", "SD", "WY",   # Great Plains
    },
    "South": {
        "AL", "AR", "GA", "KY", "LA", "MS", "NC", "SC", "TN", "TX", "VA", "WV",
        "MD", "DE",  # mid-Atlantic
    },
}

MIN_TRAIN_ROWS = 200  # skip crop×region combinations below this threshold

# LightGBM hyperparameters
LGBM_PARAMS = dict(
    n_estimators      = 1000,
    learning_rate     = 0.05,
    num_leaves        = 63,
    min_child_samples = 20,
    subsample         = 0.8,
    colsample_bytree  = 0.6,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    random_state      = 42,
    verbose           = -1,
)

# Random Forest hyperparameters
RF_PARAMS = dict(
    n_estimators    = 300,
    max_features    = 0.6,
    min_samples_leaf= 10,
    random_state    = 42,
    n_jobs          = -1,
)

# Ensemble weights [lgbm, rf]
ENSEMBLE_WEIGHTS = (0.6, 0.4)

N_CV_FOLDS   = 5
ANOMALY_Z    = 2.0
RANDOM_STATE = 42

## Data Loading

In [ ]:
if spark is not None:
    df_raw = spark.table(MERGED_TABLE).toPandas()
elif CSV_PATH and Path(CSV_PATH).exists():
    df_raw = pd.read_csv(CSV_PATH)
else:
    raise ValueError("Set spark + MERGED_TABLE, or a valid CSV_PATH")

df_raw.columns = [c.strip() for c in df_raw.columns]
df_raw["commodity"] = df_raw["commodity"].str.strip()
print(f"Loaded rows: {len(df_raw)}")

## Preprocessing & Feature Engineering

In [ ]:
def assign_region(state_abbr: str) -> str:
    s = str(state_abbr).strip().upper()
    for region, states in REGION_MAP.items():
        if s in states:
            return region
    return "Other"


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Impute missing values and engineer features. Returns a new DataFrame."""
    df = df.copy()

    # ── Numeric cast ──────────────────────────────────────────
    numeric_cols = [
        "avg_temp_max_c", "avg_temp_min_c", "avg_temp_mean_c", "peak_temp_max_c",
        "winter_snowfall_mm", "avg_snow_depth_mm", "max_snow_depth_mm", "snow_days",
        "avg_precip_mm", "precip_std_mm", "heat_stress_days", "gdd",
        "dry_days", "heavy_rain_days", "frost_days",
        "drought_flag", "flood_flag", "extreme_heat_flag",
    ]
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # ── Imputation ────────────────────────────────────────────
    for c in ["avg_temp_max_c", "avg_temp_min_c", "avg_temp_mean_c", "peak_temp_max_c"]:
        df[c] = df[c].fillna(
            df.groupby(["commodity", "state_abbr"])[c].transform("median")
        )
    for c in ["winter_snowfall_mm", "avg_snow_depth_mm", "max_snow_depth_mm", "snow_days"]:
        df[c] = df[c].fillna(0)
    df["flood_flag"]    = df["flood_flag"].fillna(0)
    df["avg_precip_mm"] = df["avg_precip_mm"].fillna(df["avg_precip_mm"].median())
    df["precip_std_mm"] = df["precip_std_mm"].fillna(df["precip_std_mm"].median())

    # ── Feature engineering ───────────────────────────────────
    df["is_irrigated"]      = (df["irrigation"].str.strip() == "Irrigated").astype(int)
    df["temp_range"]        = df["avg_temp_max_c"] - df["avg_temp_min_c"]
    df["temp_stress_index"] = df["peak_temp_max_c"] * df["heat_stress_days"]
    df["optimal_temp_score"]= -abs(df["avg_temp_mean_c"] - 22.5)
    df["precip_cv"]         = df["precip_std_mm"] / (df["avg_precip_mm"] + 1)
    df["heat_dry_combo"]    = df["heat_stress_days"] * df["dry_days"]
    # flood included in extreme_events — previously only drought+heat, missing events like 2019 Midwest flooding
    df["extreme_events"]    = df["drought_flag"] + df["extreme_heat_flag"] + df["flood_flag"]
    df["gdd_precip"]        = df["gdd"] * df["avg_precip_mm"]
    df["gdd_per_precip"]    = df["gdd"] / (df["avg_precip_mm"] + 1)
    df["year_trend"]        = df["year"] - 2010
    yt = df["year_trend"]
    df["year_x_precip"]     = yt * df["avg_precip_mm"]
    df["year_x_heat"]       = yt * df["heat_stress_days"]

    # ── Region assignment ─────────────────────────────────────
    df["region"] = df["state_abbr"].apply(assign_region)

    return df


WEATHER_FEATURES = [
    # Temperature
    "avg_temp_max_c", "avg_temp_mean_c", "peak_temp_max_c",
    "temp_range", "temp_stress_index", "optimal_temp_score",
    # Heat stress
    "heat_stress_days", "frost_days",
    # Precipitation
    "avg_precip_mm", "dry_days", "heavy_rain_days", "precip_cv",
    # Snowfall
    "winter_snowfall_mm", "snow_days",
    # Extreme weather (drought + heat + flood all included)
    "drought_flag", "extreme_heat_flag", "flood_flag", "extreme_events", "heat_dry_combo",
    # GDD
    "gdd", "gdd_precip", "gdd_per_precip",
    # Time trend
    "year_trend", "year_x_precip", "year_x_heat",
    # Irrigation
    "is_irrigated",
]

df = build_features(df_raw)
print(f"Features ready. Shape: {df.shape}  |  Feature count: {len(WEATHER_FEATURES)}")
print(df.groupby(["commodity", "region"]).size().unstack(fill_value=0))

## Helper functions

In [ ]:
def compute_metrics(actual, predicted):
    rmse = float(np.sqrt(mean_squared_error(actual, predicted)))
    mae  = float(mean_absolute_error(actual, predicted))
    r2   = float(r2_score(actual, predicted))
    return rmse, mae, r2


def add_fixed_effects(train: pd.DataFrame, test: pd.DataFrame):
    """Compute state and county baseline yields from train, apply to both splits.

    Baselines are the historical mean yield per state/county (train only).
    The model uses these as features alongside weather and year_trend,
    letting LightGBM learn the temporal trend directly without explicit detrending.
    """
    global_mean = train["yield_amount"].mean()

    state_mean = train.groupby("state_abbr")["yield_amount"].mean()
    train["state_baseline"] = train["state_abbr"].map(state_mean).fillna(global_mean)
    test["state_baseline"]  = test["state_abbr"].map(state_mean).fillna(global_mean)

    cmean = (
        train.groupby(["state_fips", "county_fips"])["yield_amount"]
        .mean().reset_index(name="county_baseline")
    )
    train = train.merge(cmean, on=["state_fips", "county_fips"], how="left")
    test  = test.merge(cmean, on=["state_fips", "county_fips"], how="left")
    train["county_baseline"] = train["county_baseline"].fillna(train["state_baseline"])
    test["county_baseline"]  = test["county_baseline"].fillna(test["state_baseline"])

    return train, test

## MLflow Training — Corn & Soybeans (year-based 8:2 split)

Train: years < 2019 / Test: years >= 2019

The model predicts `yield_amount` directly. Temporal trend is captured via `year_trend` feature — no explicit detrending needed. Anomaly detection flags residuals that weather features cannot explain.

In [ ]:
mlflow.set_experiment(experiment_id=MLFLOW_EXPERIMENT_ID)

# Year-based train/test split — prevents time leakage
df["is_test"] = df["year"] >= TEST_START_YEAR
print(f"Train years: < {TEST_START_YEAR}  ({(~df['is_test']).sum()} rows)")
print(f"Test years : >= {TEST_START_YEAR} ({df['is_test'].sum()} rows)")

FEATURES  = WEATHER_FEATURES + ["state_baseline", "county_baseline"]
w_lgbm, w_rf = ENSEMBLE_WEIGHTS
summary   = {}
models    = {}   # (crop, region) → (lgbm_model, rf_model)
test_Xy   = {}   # (crop, region) → (X_test_df, y_test_series)
train_Xy  = {}   # (crop, region) → (X_train_df, y_train_series)

with mlflow.start_run(run_name="ensemble_yield_prediction") as parent_run:
    mlflow.log_params({
        **{f"lgbm_{k}": v for k, v in LGBM_PARAMS.items()},
        **{f"rf_{k}": v for k, v in RF_PARAMS.items()},
        "ensemble_weights": str(ENSEMBLE_WEIGHTS),
        "test_start_year": TEST_START_YEAR,
        "regions": list(REGION_MAP.keys()),
    })

    for crop in ["Corn", "Soybeans"]:
        for region in REGION_MAP.keys():
            run_name = f"ensemble_{crop.lower()}_{region.lower()}"
            with mlflow.start_run(run_name=run_name, nested=True) as child_run:
                dc    = df[(df["commodity"] == crop) & (df["region"] == region)].copy()
                train = dc[~dc["is_test"]].copy()
                test  = dc[dc["is_test"]].copy()
                print(f"\n{'='*55}\n  {crop} × {region}  Train={len(train)}, Test={len(test)}\n{'='*55}")

                if len(train) < MIN_TRAIN_ROWS:
                    print(f"  SKIP: train {len(train)} < {MIN_TRAIN_ROWS}")
                    mlflow.set_tag("skipped", "insufficient_data")
                    continue

                train, test = add_fixed_effects(train, test)

                X_train_all = train[FEATURES]
                y_train_all = train["yield_amount"]
                X_test_fin  = test[FEATURES]

                # Register MLflow datasets
                train_dataset = mlflow.data.from_pandas(
                    train[["state_fips", "county_fips", "year"] + FEATURES + ["yield_amount"]],
                    name=f"train_{crop.lower()}_{region.lower()}",
                )
                test_dataset = mlflow.data.from_pandas(
                    test[["state_fips", "county_fips", "year"] + FEATURES + ["yield_amount"]],
                    name=f"test_{crop.lower()}_{region.lower()}",
                )

                # LightGBM — early stopping on 15% hold-out
                X_tr, X_val, y_tr, y_val = train_test_split(
                    X_train_all, y_train_all, test_size=0.15, random_state=RANDOM_STATE
                )
                lgbm_model = lgb.LGBMRegressor(**LGBM_PARAMS)
                lgbm_model.fit(
                    X_tr, y_tr,
                    eval_set=[(X_val, y_val)],
                    callbacks=[lgb.early_stopping(50, verbose=False)],
                )
                print(f"  LGBM best iteration: {lgbm_model.best_iteration_}")

                # Random Forest — trained on full train set
                rf_model = RandomForestRegressor(**RF_PARAMS)
                rf_model.fit(X_train_all, y_train_all)

                # Store for post-hoc analysis (feature importance, SHAP)
                models[(crop, region)]   = (lgbm_model, rf_model)
                test_Xy[(crop, region)]  = (X_test_fin.copy(), test["yield_amount"].copy())
                train_Xy[(crop, region)] = (X_train_all.copy(), y_train_all.copy())

                # Log both models
                lgbm_info = mlflow.sklearn.log_model(
                    sk_model=lgbm_model, name=f"lgbm_{crop.lower()}_{region.lower()}",
                    input_example=X_tr.head(5),
                )
                rf_info = mlflow.sklearn.log_model(
                    sk_model=rf_model, name=f"rf_{crop.lower()}_{region.lower()}",
                    input_example=X_train_all.head(5),
                )
                logged_lgbm = mlflow.get_logged_model(lgbm_info.model_id)
                logged_rf   = mlflow.get_logged_model(rf_info.model_id)

                # Train metrics (ensemble)
                train_pred = w_lgbm * lgbm_model.predict(X_train_all) + w_rf * rf_model.predict(X_train_all)
                tr_rmse, tr_mae, tr_r2 = compute_metrics(y_train_all.values, train_pred)
                for lm, ds in [(logged_lgbm, train_dataset), (logged_rf, train_dataset)]:
                    mlflow.log_metrics(
                        metrics={"train_r2": tr_r2, "train_mae": tr_mae, "train_rmse": tr_rmse},
                        model_id=lm.model_id, dataset=ds,
                    )

                # Test metrics (ensemble)
                pred_yield   = w_lgbm * lgbm_model.predict(X_test_fin) + w_rf * rf_model.predict(X_test_fin)
                actual_yield = test["yield_amount"].values
                rmse, mae, r2 = compute_metrics(actual_yield, pred_yield)
                for lm, ds in [(logged_lgbm, test_dataset), (logged_rf, test_dataset)]:
                    mlflow.log_metrics(
                        metrics={"test_r2": r2, "test_mae": mae, "test_rmse": rmse},
                        model_id=lm.model_id, dataset=ds,
                    )

                print(f"  R2={r2:.4f}  MAE={mae:.2f}  RMSE={rmse:.2f}")
                summary[(crop, region)] = {"r2": r2, "mae": mae, "rmse": rmse}

    # Log overall summary to parent run
    for (crop, region), m in summary.items():
        key = f"{crop.lower()}_{region.lower()}"
        mlflow.log_metrics({f"{key}_r2": m["r2"], f"{key}_mae": m["mae"], f"{key}_rmse": m["rmse"]})

print("\n=== Training Summary ===")
for (crop, region), m in summary.items():
    print(f"  {crop:10s} × {region:9s}: R2={m['r2']:.4f}  MAE={m['mae']:.2f}  RMSE={m['rmse']:.2f}")

## Feature Importance Dashboard

Aggregated feature importance across all crop × region models, weighted by training set size.

In [ ]:
import matplotlib.pyplot as plt

# Aggregate LightGBM feature importances across all models, weighted by training set size
# RF impurity-based importance is biased toward high-cardinality features (e.g. county_baseline)
# → use LightGBM importance for interpretation, RF is kept for prediction only
lgbm_imp = pd.Series(0.0, index=FEATURES)
total_n  = 0

for (crop, region), (lgbm_m, _) in models.items():
    n = len(train_Xy[(crop, region)][1])
    lgbm_imp += pd.Series(lgbm_m.feature_importances_, index=FEATURES) * n
    total_n  += n

lgbm_imp /= total_n
lgbm_imp /= lgbm_imp.sum()

# Split: weather features vs structural controls
structural = ["state_baseline", "county_baseline"]
weather_imp   = lgbm_imp.drop(structural).sort_values(ascending=True)
structural_imp = lgbm_imp[structural].sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 9))

# Left: weather features only
weather_imp.plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Weather Feature Importance (LightGBM)\nWhat weather variables drive yield?")
axes[0].set_xlabel("Normalized Importance")
axes[0].axvline(1 / len(weather_imp), color="red", linestyle="--", alpha=0.5, label="uniform baseline")
axes[0].legend()

# Right: all features including structural controls
lgbm_imp.sort_values(ascending=True).plot(kind="barh", ax=axes[1], color="coral")
axes[1].set_title("All Features (LightGBM)\nIncluding structural controls")
axes[1].set_xlabel("Normalized Importance")

plt.suptitle("Feature Importance Dashboard — LightGBM\n"
             "(RF omitted: impurity-based importance overestimates high-cardinality features)",
             fontsize=12)
plt.tight_layout()
plt.show()

print("\nTop 10 weather features (LightGBM):")
print(weather_imp.sort_values(ascending=False).head(10).to_string())
print(f"\nStructural controls:")
print(structural_imp.sort_values(ascending=False).to_string())

## 5-Fold CV Anomaly Detection

In [ ]:
# CV uses county-based split — evaluates spatial generalization across all years
county_ids = df[["state_fips", "county_fips"]].drop_duplicates().reset_index(drop=True)
kf = KFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

df["pred_yield"] = np.nan
w_lgbm, w_rf = ENSEMBLE_WEIGHTS

with mlflow.start_run(run_name=f"ensemble_{N_CV_FOLDS}fold_cv_anomaly") as cv_run:
    mlflow.log_params({
        **{f"lgbm_{k}": v for k, v in LGBM_PARAMS.items()},
        **{f"rf_{k}": v for k, v in RF_PARAMS.items()},
        "ensemble_weights": str(ENSEMBLE_WEIGHTS),
        "n_cv_folds":          N_CV_FOLDS,
        "anomaly_z_threshold": ANOMALY_Z,
        "regions":             list(REGION_MAP.keys()),
    })

    fold_r2s = []

    for fold, (train_county_idx, test_county_idx) in enumerate(kf.split(county_ids)):
        test_counties = set(map(tuple, county_ids.iloc[test_county_idx].values))
        key       = list(zip(df["state_fips"], df["county_fips"]))
        test_mask = np.array([k in test_counties for k in key])

        fold_preds, fold_actuals = [], []

        for crop in ["Corn", "Soybeans"]:
            for region in REGION_MAP.keys():
                mask = (df["commodity"] == crop) & (df["region"] == region)
                tr = df[~test_mask & mask].copy()
                te = df[ test_mask & mask].copy()

                if len(tr) < MIN_TRAIN_ROWS or len(te) == 0:
                    continue

                tr, te = add_fixed_effects(tr, te)

                FEATURES = WEATHER_FEATURES + ["state_baseline", "county_baseline"]

                # LightGBM — early stopping on 15% hold-out
                val_idx   = tr.sample(frac=0.15, random_state=RANDOM_STATE).index
                tr_inner  = tr.loc[~tr.index.isin(val_idx)]
                val_inner = tr.loc[val_idx]

                lgbm_model = lgb.LGBMRegressor(**LGBM_PARAMS)
                lgbm_model.fit(
                    tr_inner[FEATURES], tr_inner["yield_amount"],
                    eval_set=[(val_inner[FEATURES], val_inner["yield_amount"])],
                    callbacks=[lgb.early_stopping(50, verbose=False)],
                )

                # Random Forest — full tr
                rf_model = RandomForestRegressor(**RF_PARAMS)
                rf_model.fit(tr[FEATURES], tr["yield_amount"])

                # Ensemble prediction
                pred_yield = (
                    w_lgbm * lgbm_model.predict(te[FEATURES]) +
                    w_rf   * rf_model.predict(te[FEATURES])
                )
                df.loc[te.index, "pred_yield"] = pred_yield

                fold_preds.extend(pred_yield.tolist())
                fold_actuals.extend(te["yield_amount"].values.tolist())

        if fold_actuals:
            fold_r2 = r2_score(fold_actuals, fold_preds)
            fold_r2s.append(fold_r2)
            mlflow.log_metric("fold_r2", fold_r2, step=fold + 1)
            print(f"  Fold {fold+1}/{N_CV_FOLDS} done — R2={fold_r2:.4f}")

    # ── Residual → Z-score → Anomaly flag ─────────────────────
    valid_mask = df["pred_yield"].notna()
    df.loc[valid_mask, "residual"] = (
        df.loc[valid_mask, "pred_yield"] - df.loc[valid_mask, "yield_amount"]
    ).abs()

    df["residual_zscore"] = df.groupby("commodity")["residual"].transform(
        lambda x: (x - x.mean()) / x.std()
    )
    df["anomaly_flag"] = (df["residual_zscore"] > ANOMALY_Z).astype(int)

    total_anomalies = int(df["anomaly_flag"].sum())
    anomaly_pct     = float(df["anomaly_flag"].mean() * 100)
    cv_r2_mean      = float(np.mean(fold_r2s)) if fold_r2s else float("nan")

    mlflow.log_metrics({
        "cv_r2_mean":      cv_r2_mean,
        "total_anomalies": total_anomalies,
        "anomaly_pct":     anomaly_pct,
    })

    print(f"\nCV R2 mean : {cv_r2_mean:.4f}")
    print(f"Anomalies  : {total_anomalies} ({anomaly_pct:.1f}%)")

## Weather Anomaly Detection

Flags county-years where weather features themselves are statistically extreme (Z-score > 2.0).

| anomaly_type | meaning |
|---|---|
| **Extreme Weather → Yield Impact** | unusual weather AND model failed to predict → high confidence weather event |
| **Yield Mismatch (unexplained)** | model failed but weather was normal → data error or non-weather factor |
| **Extreme Weather (model handled)** | unusual weather but model predicted correctly |
| **Normal** | no flags |

In [ ]:
WEATHER_ANOMALY_Z = 2.0

# Raw weather features to monitor for extremity (interpretable ones)
WEATHER_RAW = [c for c in [
    "avg_precip_mm", "heat_stress_days", "drought_flag", "flood_flag",
    "extreme_heat_flag", "gdd", "frost_days", "dry_days", "heavy_rain_days",
    "avg_temp_mean_c", "peak_temp_max_c",
] if c in df.columns]

# Z-scores per commodity (normalize within crop type)
for feat in WEATHER_RAW:
    df[f"wz_{feat}"] = df.groupby("commodity")[feat].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )

wz_cols = [f"wz_{f}" for f in WEATHER_RAW]
df["weather_extremity"]    = df[wz_cols].abs().max(axis=1)
df["weather_anomaly_flag"] = (df["weather_extremity"] > WEATHER_ANOMALY_Z).astype(int)

# Combined anomaly type
conditions = [
    (df["anomaly_flag"] == 1) & (df["weather_anomaly_flag"] == 1),
    (df["anomaly_flag"] == 1) & (df["weather_anomaly_flag"] == 0),
    (df["anomaly_flag"] == 0) & (df["weather_anomaly_flag"] == 1),
]
choices = [
    "Extreme Weather → Yield Impact",
    "Yield Mismatch (unexplained)",
    "Extreme Weather (model handled)",
]
df["anomaly_type"] = np.select(conditions, choices, default="Normal")

print("=== Anomaly Type Distribution ===")
print(df["anomaly_type"].value_counts().to_string())

print("\n=== Extreme Weather Events by Year ===")
extreme = df[df["weather_anomaly_flag"] == 1]
print(extreme.groupby(["year", "commodity"])["weather_anomaly_flag"].count().unstack(fill_value=0).to_string())

print("\n=== Most Extreme Weather Counties ===")
top_extreme = (
    df[df["weather_anomaly_flag"] == 1]
    .groupby(["state_abbr", "county_fips", "year"])
    .agg(weather_extremity=("weather_extremity", "max"), commodity=("commodity", "first"))
    .reset_index()
    .sort_values("weather_extremity", ascending=False)
    .head(15)
)
display(top_extreme)

## SHAP Explainer — What Drives Yield?

SHAP (SHapley Additive exPlanations) shows:
1. **Bar plot** — which features matter most on average
2. **Beeswarm** — how each feature pushes predictions up or down
3. **Waterfall** — why the model failed most on a specific county-year

In [ ]:
import shap

# Use the model with most test data for SHAP analysis
target = max(test_Xy.keys(), key=lambda k: len(test_Xy[k][1]))
target_crop, target_region = target
lgbm_m, _ = models[target]
X_test_df, y_test = test_Xy[target]

print(f"SHAP analysis: {target_crop} × {target_region}  ({len(X_test_df)} test rows)")

explainer   = shap.TreeExplainer(lgbm_m)
shap_values = explainer.shap_values(X_test_df)

# 1. Global importance (bar)
print("\n[1] Global Feature Importance — SHAP")
shap.summary_plot(shap_values, X_test_df, plot_type="bar", show=True,
                  plot_size=(10, 7))

# 2. Beeswarm — direction and magnitude per feature
print("\n[2] Feature Impact Distribution — SHAP Beeswarm")
shap.summary_plot(shap_values, X_test_df, show=True, plot_size=(10, 7))

# 3. Waterfall for the single worst-predicted row (highest residual)
pred_test   = lgbm_m.predict(X_test_df)
residuals   = np.abs(pred_test - y_test.values)
worst_pos   = int(np.argmax(residuals))
worst_info  = X_test_df.iloc[worst_pos]

print(f"\n[3] Worst Prediction Explained")
print(f"    Actual={y_test.values[worst_pos]:.1f}  Predicted={pred_test[worst_pos]:.1f}  "
      f"Residual={residuals[worst_pos]:.1f}")

shap.waterfall_plot(shap.Explanation(
    values        = shap_values[worst_pos],
    base_values   = explainer.expected_value,
    data          = worst_info.values,
    feature_names = FEATURES,
))

## Map Visualization

County-level choropleth of yield anomaly rate and extreme weather rate.
Grey counties = no corn/soybean data in the dataset.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

try:
    from pygris import counties as get_counties
    us_counties = get_counties(year=2020, cache=True)
    us_counties["state_fips"]  = us_counties["STATEFP"].astype(int)
    us_counties["county_fips"] = us_counties["COUNTYFP"].astype(int)
    us_counties = us_counties[us_counties["state_fips"] < 60]  # exclude territories

    county_agg = (
        df[df["pred_yield"].notna()]
        .groupby(["state_fips", "county_fips"])
        .agg(
            anomaly_rate         = ("anomaly_flag",         "mean"),
            weather_anomaly_rate = ("weather_anomaly_flag", "mean"),
            avg_residual_zscore  = ("residual_zscore",      "mean"),
        )
        .reset_index()
    )

    gdf = us_counties.merge(county_agg, on=["state_fips", "county_fips"], how="left")

    # --- Map 1: Yield Anomaly Rate ---
    fig, ax = plt.subplots(figsize=(24, 14))
    gdf.plot(
        column="anomaly_rate", ax=ax,
        cmap="Reds", legend=True,
        legend_kwds={
            "label": "Yield Anomaly Rate",
            "orientation": "horizontal",
            "shrink": 0.5,
            "pad": 0.02,
            "fraction": 0.03,
        },
        missing_kwds={"color": "lightgrey", "label": "No data"},
        linewidth=0.1, edgecolor="white",
    )
    ax.set_title("Yield Anomaly Rate by County\n(fraction of years with unexplained yield deviation)",
                 fontsize=18, pad=16)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    # --- Map 2: Extreme Weather Rate ---
    fig, ax = plt.subplots(figsize=(24, 14))
    gdf.plot(
        column="weather_anomaly_rate", ax=ax,
        cmap="Blues", legend=True,
        legend_kwds={
            "label": "Extreme Weather Rate",
            "orientation": "horizontal",
            "shrink": 0.5,
            "pad": 0.02,
            "fraction": 0.03,
        },
        missing_kwds={"color": "lightgrey", "label": "No data"},
        linewidth=0.1, edgecolor="white",
    )
    ax.set_title("Extreme Weather Rate by County\n(fraction of years with extreme weather conditions)",
                 fontsize=18, pad=16)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"Map rendering failed: {e}")
    print("Fallback: state-level anomaly summary")
    state_agg = (
        df[df["pred_yield"].notna()]
        .groupby("state_abbr")
        .agg(anomaly_rate=("anomaly_flag", "mean"), weather_anomaly_rate=("weather_anomaly_flag", "mean"))
        .sort_values("anomaly_rate", ascending=False)
    )
    print(state_agg.head(20).to_string())

## Anomaly Summary

In [ ]:
print("Anomaly summary by crop:")
print(df.groupby("commodity")["anomaly_flag"].agg(["sum", "mean", "count"]))

out_cols = [
    "state_fips", "county_fips", "year", "county_name", "state_abbr",
    "commodity", "irrigation", "yield_amount", "pred_yield",
    "residual", "residual_zscore", "anomaly_flag",
]
result_df = df[[c for c in out_cols if c in df.columns]].copy()

display(result_df[result_df["anomaly_flag"] == 1].sort_values("residual_zscore", ascending=False).head(20))